# 1. RBAC and Custom Roles

AZ-500 expects you to **implement** role-based access control (RBAC) — not just know what it is. You create custom roles, assign them at the right scope, and understand the permission model.

## Before you run this notebook

1. Run `uv sync` in the lab folder.
2. In VS Code, click the **kernel picker** at the top-right of this notebook and choose the interpreter from `.venv` (the one created by `uv`).
3. If the kernel isn't listed, reload the window (`Cmd+Shift+P` then *Reload Window*).

No Docker is needed for this notebook — everything runs as plain Python.

## RBAC essentials

Azure RBAC has four elements:

| Element | What it is | Example |
|---------|-----------|----------|
| **Security principal** | Who gets access | User, group, service principal, managed identity |
| **Role definition** | What they can do | `Actions`, `NotActions`, `DataActions`, `NotDataActions` |
| **Scope** | Where it applies | Management group -> Subscription -> Resource group -> Resource |
| **Assignment** | Binding principal + role + scope | *"Alice is Contributor on rg-prod"* |


In [ ]:
import json

# Azure RBAC scope hierarchy. Roles assigned higher are INHERITED downward.
SCOPE_HIERARCHY = {
    'Management Group: Contoso': {
        'Subscription: Production': {
            'Resource Group: rg-web':  ['App Service: web-app', 'SQL DB: web-db', 'Key Vault: web-kv'],
            'Resource Group: rg-data': ['Storage: datalake', 'Synapse: analytics'],
        },
        'Subscription: Development': {
            'Resource Group: rg-dev':  ['App Service: dev-app', 'SQL DB: dev-db'],
        },
    },
}

def show_scope(scope, indent=0):
    prefix = '  ' * indent
    if isinstance(scope, dict):
        for k, v in scope.items():
            print(f'{prefix}[FOLDER] {k}')
            show_scope(v, indent + 1)
    elif isinstance(scope, list):
        for item in scope:
            print(f'{prefix}  - {item}')

print('=== Azure RBAC scope hierarchy ===')
print('Roles at a higher scope are INHERITED by all children.\n')
show_scope(SCOPE_HIERARCHY)
print('\nIf Alice is "Contributor" on the Production subscription,')
print('she has Contributor on EVERY resource in rg-web and rg-data.')


## Built-in roles

Azure ships **hundreds** of built-in roles. The ones the exam keeps coming back to:

| Role | Permissions | Typical use |
|------|-------------|-------------|
| **Owner** | `Actions: *`, no `NotActions`. Everything, **including** assigning roles. | Subscription admins |
| **Contributor** | `Actions: *` minus `Microsoft.Authorization/*/Write`, `Microsoft.Authorization/*/Delete` and `elevateAccess`. | DevOps teams |
| **Reader** | `Actions: */read` | Auditors |
| **User Access Administrator** | `*/read` + **all** of `Microsoft.Authorization/*` + `Microsoft.Support/*` | Delegating access management |
| **Role Based Access Control Administrator** | `*/read` + `roleAssignments` write and delete + `Microsoft.Support/*` | Least-privilege alternative to User Access Administrator |
| **Key Vault Administrator** | All Key Vault **data-plane** operations. Cannot manage the vault resource. | Ops teams |
| **Key Vault Secrets User** | `getSecret` + `readMetadata` (data plane) | Applications |
| **Storage Blob Data Contributor** | Read/write/delete blob **data** | Apps accessing storage |
| **Network Contributor** | Manage networking | Network team |

Three traps hiding in that table:

- **Contributor can still *read* role assignments.** Its `NotActions` cover
  `Microsoft.Authorization/*/Write` and `Microsoft.Authorization/*/Delete` — not
  `*/read`. Contributor can see who has what; it just cannot change it, and it cannot
  call `elevateAccess` to make itself User Access Administrator at tenant root.
- **User Access Administrator is not "role assignments only".** `Microsoft.Authorization/*`
  also covers policy assignments, locks and role *definitions*, and `*/read` lets it read
  every resource in scope. If all you want to delegate is handing out role assignments,
  use **Role Based Access Control Administrator** instead — that is the modern
  least-privilege answer.
- **Key Vault Administrator is not Key Vault Contributor.** Administrator is the *data*
  plane: read and write every secret, key and certificate, but it cannot change the vault
  resource (firewall, delete, purge protection). Key Vault Contributor is the mirror image
  — it manages the vault and can read none of the data in it.

### Control plane vs data plane

This distinction is critical for AZ-500:

- **Control plane** (`Actions` / `NotActions`): manage the resource itself (create, delete, configure).
- **Data plane** (`DataActions` / `NotDataActions`): access the data *inside* the resource (read blobs, get secrets).

**Neither Owner nor Contributor has a single `DataAction`.** Straight out of the built-in
role JSON, both are `"dataActions": []`. So neither can read a Key Vault secret or the bytes
of a blob — that is why you still need `Key Vault Secrets User` or
`Storage Blob Data Contributor` on top. The difference between the two is what happens next:
Owner can *assign itself* the data role and then read everything, Contributor cannot.
The data-plane gap protects you from accidents, not from a malicious Owner.


In [ ]:
import json
from fnmatch import fnmatchcase

# A tiny simulation of Azure's permission engine, using the real built-in role
# definitions (trimmed to the entries the exam cares about).
ROLES = {
    'Owner': {
        'Actions': ['*'],
        'NotActions': [],
        'DataActions': [],       # <- Owner has NO data-plane permissions. Really.
        'NotDataActions': [],
    },
    'Contributor': {
        'Actions': ['*'],
        # Straight from the built-in definition. Note it denies Write and Delete
        # under Microsoft.Authorization - NOT read.
        'NotActions': [
            'Microsoft.Authorization/*/Write',
            'Microsoft.Authorization/*/Delete',
            'Microsoft.Authorization/elevateAccess/Action',
            'Microsoft.Resources/deploymentStacks/manageDenySetting/action',
        ],
        'DataActions': [],
        'NotDataActions': [],
    },
    'User Access Administrator': {
        'Actions': ['*/read', 'Microsoft.Authorization/*', 'Microsoft.Support/*'],
        'NotActions': [],
        'DataActions': [],
        'NotDataActions': [],
    },
    'Storage Blob Data Contributor': {
        'Actions': [
            'Microsoft.Storage/storageAccounts/blobServices/containers/read',
            'Microsoft.Storage/storageAccounts/blobServices/containers/write',
            'Microsoft.Storage/storageAccounts/blobServices/containers/delete',
        ],
        'NotActions': [],
        'DataActions': ['Microsoft.Storage/storageAccounts/blobServices/containers/blobs/*'],
        'NotDataActions': [],
    },
    'Key Vault Secrets User': {
        'Actions': [],
        'NotActions': [],
        'DataActions': [
            'Microsoft.KeyVault/vaults/secrets/getSecret/action',
            'Microsoft.KeyVault/vaults/secrets/readMetadata/action',
        ],
        'NotDataActions': [],
    },
}

def _matches(pattern: str, op: str) -> bool:
    # Azure action strings are case-INsensitive and '*' is a glob that may appear
    # anywhere and may span '/' - which is exactly what the real Contributor
    # definition relies on ('Microsoft.Authorization/*/Write').
    return fnmatchcase(op.lower(), pattern.lower())

def check_permission(role_name: str, operation: str, plane: str) -> str:
    role = ROLES[role_name]
    if plane == 'control':
        actions, not_actions = role['Actions'], role['NotActions']
    else:
        actions, not_actions = role['DataActions'], role['NotDataActions']
    if any(_matches(p, operation) for p in not_actions):
        return 'DENIED (NotActions)'
    if any(_matches(p, operation) for p in actions):
        return 'ALLOWED'
    return 'NOT PERMITTED (no matching action)'

print('=== Permission checks ===\n')
checks = [
    ('Owner', 'Microsoft.Authorization/roleAssignments/write', 'control', 'Assign a role'),
    ('Owner', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Contributor', 'Microsoft.Compute/virtualMachines/delete', 'control', 'Delete a VM'),
    ('Contributor', 'Microsoft.Authorization/roleAssignments/write', 'control', 'Assign a role'),
    ('Contributor', 'Microsoft.Authorization/roleAssignments/read', 'control', 'READ role assignments'),
    ('Contributor', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('User Access Administrator', 'Microsoft.Authorization/roleAssignments/write', 'control', 'Assign a role'),
    ('User Access Administrator', 'Microsoft.Compute/virtualMachines/write', 'control', 'Create a VM'),
    ('Key Vault Secrets User', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data', 'Read a Key Vault secret'),
    ('Storage Blob Data Contributor',
     'Microsoft.Storage/storageAccounts/blobServices/containers/blobs/read', 'data', 'Read a blob'),
]
for role, op, plane, desc in checks:
    result = check_permission(role, op, plane)
    print(f'{result:<28}  {role:<28}  ->  {desc}')

# Pin the four facts this notebook teaches, so an edit to the role definitions above
# can never quietly stop demonstrating them.
assert check_permission('Contributor', 'Microsoft.Authorization/roleAssignments/write',
                        'control').startswith('DENIED'), 'Contributor must not be able to assign roles'
assert check_permission('Contributor', 'Microsoft.Authorization/roleAssignments/read',
                        'control') == 'ALLOWED', \
    'Contributor CAN read role assignments - only Write and Delete are in NotActions'
for privileged in ('Owner', 'Contributor'):
    assert check_permission(privileged, 'Microsoft.KeyVault/vaults/secrets/getSecret/action',
                            'data') != 'ALLOWED', \
        f'{privileged} has no DataActions, so it cannot read a secret without a data-plane role'
assert check_permission('User Access Administrator', 'Microsoft.Compute/virtualMachines/write',
                        'control') != 'ALLOWED', \
    'User Access Administrator manages access, not resources'
print('\nAll RBAC invariants hold.')


## Bad to Best: least privilege in action

Real-world scenario: *"Alice is on the VM support team. She needs to restart VMs in the production resource group when there is an incident."*

Let's see three approaches, from worst to best, and check each one against the principle of **least privilege**.


In [ ]:
# The custom role we are about to write as JSON. Registering it in ROLES here means
# the comparison table below and the JSON in the next code cell cannot drift apart.
VM_RESTART_ACTIONS = [
    'Microsoft.Compute/virtualMachines/read',
    'Microsoft.Compute/virtualMachines/restart/action',
    'Microsoft.Compute/virtualMachines/start/action',
    'Microsoft.Compute/virtualMachines/powerOff/action',
    'Microsoft.Resources/subscriptions/resourceGroups/read',
]
ROLES['VM Restart Operator'] = {
    'Actions': VM_RESTART_ACTIONS,
    'NotActions': [],
    'DataActions': [],
    'NotDataActions': [],
}

# Each row is (label, role, scope). Lower privilege is better.
SCENARIOS = [
    ('BAD:    Owner on subscription',            'Owner',       '/subscriptions/SUB'),
    ('OKAY:   Contributor on resource group',    'Contributor', '/subscriptions/SUB/resourceGroups/rg-prod'),
    ('BEST:   Custom "VM Restart Operator" on resource group',
                                        'VM Restart Operator', '/subscriptions/SUB/resourceGroups/rg-prod'),
]

# What can Alice do with each role? COMPUTED from the same engine as the cell above -
# a hand-written yes/no table is exactly the thing that silently drifts from the role JSON.
CAPABILITY_OPS = [
    ('View VM',         'Microsoft.Compute/virtualMachines/read',             'control'),
    ('Restart',         'Microsoft.Compute/virtualMachines/restart/action',   'control'),
    ('Delete',          'Microsoft.Compute/virtualMachines/delete',           'control'),
    ('Create',          'Microsoft.Compute/virtualMachines/write',            'control'),
    ('Assign roles',    'Microsoft.Authorization/roleAssignments/write',      'control'),
    ('Read KV secrets', 'Microsoft.KeyVault/vaults/secrets/getSecret/action', 'data'),
]

def granted(role):
    return {label for label, op, plane in CAPABILITY_OPS
            if check_permission(role, op, plane) == 'ALLOWED'}

header = f'{"Approach":<62} ' + '  '.join(f'{l:<15}' for l, _op, _pl in CAPABILITY_OPS)
print(header)
print('-' * len(header))
for label, role, _scope in SCENARIOS:
    caps = granted(role)
    row = '  '.join(f'{("yes" if l in caps else "no"):<15}' for l, _op, _pl in CAPABILITY_OPS)
    print(f'{label:<62} {row}')

# Least privilege has to shrink monotonically down the list, or the "bad -> best"
# story is not a story.
allowed = {role: granted(role) for _label, role, _scope in SCENARIOS}
assert allowed['VM Restart Operator'] < allowed['Contributor'] < allowed['Owner'], \
    f'privilege must strictly shrink at each step, got {allowed}'
assert 'Restart' in allowed['VM Restart Operator'], 'the custom role must still let Alice do her job'
assert 'Delete' not in allowed['VM Restart Operator'], 'the custom role must not be able to delete VMs'
assert 'Assign roles' not in allowed['Contributor'], 'Contributor must not be able to assign roles'

print('''
Takeaway:
  * Owner gives Alice the power to DELETE production VMs and to reassign roles.
  * Contributor still lets her delete or recreate VMs - one typo could cause an outage.
  * A narrow custom role scoped to rg-prod gives her exactly what she needs and nothing more.
    If her account is ever compromised, the blast radius is tiny.

  Look at the "Read KV secrets" column: it is "no" for ALL THREE, because Owner and
  Contributor have no DataActions at all. Owner is still the dangerous one - it can
  assign itself "Key Vault Secrets User" and come back a second later as a reader.
''')


## Custom role JSON definition

When no built-in role fits, define a custom role. This is the JSON Azure expects.

`AssignableScopes` is the property people get wrong. It is not "where this role is
assigned" — it is "where this role is *available* to be assigned", and it also controls
who may edit the role definition (you need `Microsoft.Authorization/roleDefinitions/write`
on **every** scope listed). The documented limits:

| Limit | Value |
|-------|-------|
| Custom roles per tenant | 5,000 |
| `AssignableScopes` entries | up to 2,000, but **at most one management group** |
| Root scope `"/"` | not allowed |
| Wildcards in `AssignableScopes` | not allowed (they would let you widen your own access) |
| Custom role with `DataActions` | cannot be assigned at management group scope |
| Deleting a role that is still assigned | blocked (`RoleDefinitionHasAssignments`) |


In [ ]:
custom_role = {
    'Name': 'VM Restart Operator',
    'Description': 'View and restart VMs. Cannot create or delete them.',
    'Actions': VM_RESTART_ACTIONS,
    'NotActions': [],
    'DataActions': [],
    'NotDataActions': [],
    'AssignableScopes': ['/subscriptions/00000000-0000-0000-0000-000000000000'],
}

# The JSON you ship and the role we simulated above must be the same role.
assert custom_role['Actions'] == ROLES['VM Restart Operator']['Actions']

print('Custom role JSON:')
print(json.dumps(custom_role, indent=2))

print('\n--- Azure CLI: create the role ---')
print('az role definition create --role-definition @vm-restart-operator.json')

print('\n--- Azure CLI: assign the role ---')
print('az role assignment create \\')
print('  --assignee alice@contoso.com \\')
print('  --role "VM Restart Operator" \\')
print('  --scope /subscriptions/.../resourceGroups/rg-prod')

# Quick sanity check: run the shipped JSON through the permission engine rather than
# eyeballing the Actions list.
print('\n--- What Alice can and cannot do with this role ---')
tests = [
    ('Microsoft.Compute/virtualMachines/read',            'View VMs',          True),
    ('Microsoft.Compute/virtualMachines/restart/action',  'Restart VMs',       True),
    ('Microsoft.Compute/virtualMachines/delete',          'Delete VMs',        False),
    ('Microsoft.Compute/virtualMachines/write',           'Create/update VMs', False),
]
for op, desc, expected in tests:
    allowed_here = check_permission('VM Restart Operator', op, 'control') == 'ALLOWED'
    assert allowed_here == expected, f'{desc}: expected allowed={expected}'
    print(f'  {"yes" if allowed_here else "no":<4} {desc}')


## Access Reviews - keep permissions fresh

Permissions accumulate over time ("privilege creep"). **Access Reviews** (part of
**Microsoft Entra ID Governance**, licensed with Microsoft Entra ID P2 or an Entra ID
Governance licence) periodically ask an owner: *"Does Alice still need Contributor on rg-prod?"*

Typical exam-worthy setup:

| Setting | Recommended value |
|---------|-------------------|
| **Scope** | Privileged roles, guest users, high-sensitivity groups |
| **Reviewers** | Group owner, resource owner, or the user themselves ("self-review") |
| **Frequency** | Quarterly for standing access, monthly for privileged |
| **Auto-apply results** | Yes - remove access if not approved |
| **If reviewer doesn't respond** | Remove access (safer default) |

```bash
# Create an access review for the "Contributor" role on a subscription.
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/identityGovernance/accessReviews/definitions' \
  --body @access-review.json
```

## Deny assignments

Azure also supports **deny assignments** that *block* access even if a role assignment
grants it. **Deny always wins over allow** — a deny assignment beats Owner.

You **cannot create a deny assignment directly**; Azure creates and manages them
(`IsSystemProtected` is `true` on all of them today). They appear when:

- you set `denySettings` on a **deployment stack** (the modern way, and the one to know);
- an **Azure managed application** protects its managed resource group;
- an **Azure Blueprints** assignment applies a resource lock — the legacy source.
  Blueprints is retiring, and Microsoft's migration target is deployment stacks, so
  don't reach for it in a new design.

Two properties worth remembering because they have no equivalent on a role assignment:

| Property | What it does |
|----------|--------------|
| `ExcludePrincipals` | Deny *everyone except* these principals — combined with the system-defined **All Principals**, this is how "nobody but the stack can delete this" is expressed. |
| `DoNotApplyToChildScopes` | A deny assignment can be made **non-inheriting**. Role assignments always inherit downward; deny assignments do not have to. |

---
## Summary

| Concept | What to remember |
|---------|-------------------|
| **Scope hierarchy** | MG -> Sub -> RG -> Resource. Role assignments inherit downward. |
| **Control vs data plane** | `Actions` manages the resource. `DataActions` accesses the data inside. |
| **Owner / Contributor gap** | Both have zero `DataActions`. Add `Key Vault Secrets User` etc. Owner can grant itself that role; Contributor cannot. |
| **Contributor vs UAA** | Contributor can *read* role assignments but not write them. UAA gets all of `Microsoft.Authorization/*` plus `*/read`; prefer **RBAC Administrator** if you only need role assignments. |
| **Key Vault Administrator** | Data plane only. Managing the vault resource is Key Vault Contributor. |
| **Least privilege** | Prefer a narrow custom role at the smallest scope over a broad built-in role. |
| **Access Reviews** | Fight privilege creep. Entra ID Governance / P2. Quarterly for standing access, monthly for privileged. |
| **Deny assignments** | Always win, cannot be created by you, can exclude principals, and can opt out of inheritance. |


---
## ✅ Self-check

Answer these before moving on. Answers are in the next cell.

1. Bob has **Contributor** on `rg-prod`. He opens Access control (IAM) on a VM in that
   resource group. Can he *see* the existing role assignments? Can he add one?
2. Carol has **Owner** on the subscription. The Key Vault `web-kv` uses the Azure RBAC
   permission model. Can Carol run `az keyvault secret show` on it right now? Is `web-kv`
   safe from her?
3. Dave needs to hand out role assignments on `rg-web` and nothing else. Which built-in
   role, and why not User Access Administrator?
4. A role assignment gives Erin `Owner` on the subscription. A deny assignment on one
   storage account denies `Microsoft.Storage/*/delete`. Can Erin delete that storage
   account?
5. You assign **Reader** at the management group and **Contributor** at `rg-dev`.
   What are Frank's effective permissions in `rg-dev`?
6. Your custom role's `AssignableScopes` is a single subscription. Can you assign that
   role at a resource group inside a *different* subscription in the same tenant?
7. An app can list the blob containers in a storage account but gets 403 reading a blob.
   Which of `Actions` / `DataActions` is missing, and which built-in role fixes it?


In [ ]:
answers = """
1. SEE yes, ADD no. Contributor's NotActions are Microsoft.Authorization/*/Write and
   Microsoft.Authorization/*/Delete. Reads are not excluded, so roleAssignments/read is
   allowed. This is a favourite exam distractor - "Contributor cannot touch role
   assignments" is wrong; it cannot CHANGE them.

2. NOT RIGHT NOW, and NO it is not safe. Owner is Actions ['*'] with dataActions [].
   getSecret is a DataAction, so the call fails today. But Owner can assign itself
   'Key Vault Secrets User' on the vault and retry ten seconds later. The data-plane
   split is a guardrail against mistakes, not a boundary against a determined Owner.
   (Use PIM so nobody is a standing Owner - notebook 2.)

3. Role Based Access Control Administrator, scoped to rg-web. User Access Administrator
   also grants */read over everything in scope plus ALL of Microsoft.Authorization/* -
   policy assignments, locks, role definitions - which is far more than "hand out roles".

4. NO. Deny assignments beat every role assignment, including Owner. That is the whole
   point of the deployment-stack denySettings that create them. Erin would have to be in
   the deny assignment's ExcludePrincipals list.

5. Contributor. Assignments are ADDITIVE and inherit downward, so Frank holds the union:
   Reader (from the MG) plus Contributor (on rg-dev). Reader adds nothing Contributor
   does not already have. There is no "closest assignment wins" rule in Azure RBAC -
   the only thing that subtracts is a deny assignment.

6. NO. A role can only be assigned at or below a scope listed in AssignableScopes.
   Add the second subscription to AssignableScopes, or list a management group above
   both - but only ONE management group is allowed, AssignableScopes can never be the
   root scope "/", and it cannot contain a wildcard. If the role had DataActions you
   could not assign it at the management group scope at all, only at subscriptions
   inside it.

7. DataActions. Listing containers is the CONTROL plane
   (Microsoft.Storage/storageAccounts/blobServices/containers/read); reading the bytes of
   a blob is the DATA plane (.../containers/blobs/read). Add Storage Blob Data Reader for
   read-only, or Storage Blob Data Contributor for read/write.
"""
print(answers)


**Next**: [Notebook 2 - PIM and Conditional Access](02_pim_and_conditional_access.ipynb)
